# Test Pinecone Embeddings / Indexing

This notebook tests `rag/embeddings.py`.

Important: Pinecone does the actual embedding for this project. Our Python code prepares chunk records and sends them to Pinecone.

In [ ]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

# Make imports work whether VS Code opens the project folder or the notebook folder.
project_root = Path.cwd()
if not (project_root / "rag").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
load_dotenv(project_root / ".env")

print(f"Project root: {project_root}")

## 1. Check settings

This checks whether the required `.env` values exist. It does not print your API key.

In [ ]:
required_settings = [
    "PINECONE_API_KEY",
    "PINECONE_INDEX_HOST",
    "PINECONE_TEXT_FIELD",
    "PINECONE_NAMESPACE_FIXED",
    "PINECONE_NAMESPACE_PARAGRAPH",
]

for setting in required_settings:
    value = os.getenv(setting, "")
    status = "ready" if value and "your_" not in value else "missing or placeholder"
    print(f"{setting}: {status}")

## 2. Convert a chunk into a Pinecone record

This does not call Pinecone. It only tests that our chunk is packaged correctly.

In [ ]:
from rag.embeddings import chunk_to_record, chunks_to_records

sample_chunk = {
    "chunk_id": "demo-doc:chunk:1",
    "text": "This is a sample document chunk for testing Pinecone integrated embedding.",
    "document_name": "demo.txt",
    "file_type": ".txt",
    "source_label": "Paragraph 1",
    "chunk_number": 1,
    "chunking_strategy": "fixed_size",
    "chunk_size_words": 80,
    "overlap_words": 20,
    "word_count": 10,
}

record = chunk_to_record(sample_chunk)
records = chunks_to_records([sample_chunk])

print(record)
print(f"Number of records: {len(records)}")

## 3. Optional: upload one test chunk to Pinecone

Leave `RUN_PINECONE_UPLOAD = False` if you only want a local test. Set it to `True` when you are ready to create a real record in Pinecone.

In [ ]:
from rag.embeddings import index_chunks

RUN_PINECONE_UPLOAD = False
namespace = os.getenv("PINECONE_NAMESPACE_FIXED", "fixed_chunks")

if RUN_PINECONE_UPLOAD:
    uploaded_count = index_chunks([sample_chunk], namespace=namespace)
    print(f"Uploaded {uploaded_count} record(s) to namespace: {namespace}")
else:
    print("Upload skipped. Set RUN_PINECONE_UPLOAD = True when you want to test Pinecone.")

## 4. Optional: search Pinecone

Run this after uploading a test chunk or after indexing real document chunks.

In [ ]:
from rag.embeddings import search_chunks

RUN_PINECONE_SEARCH = False
top_k = 3  # Change this to the number of chunks you want Pinecone to return.

if RUN_PINECONE_SEARCH:
    results = search_chunks(
        question="What is this document chunk about?",
        namespace=namespace,
        top_k=top_k,
    )
    results
else:
    print("Search skipped. Set RUN_PINECONE_SEARCH = True after uploading records.")